<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_05_window_scaling_seq2one/stage_05_window_scaling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **stage_05_window_scaling**




## **0. Configuración del Entorno**


### 0.1. Acceso a Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


### 0.2. Instalación de librerías


In [2]:
#!{sys.executable} -m pip install -q ta
#print("Librería instalada: technical-analysis")

### 0.3. Importación de librerías


In [3]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos
#import ta
import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal

#from ta.momentum import StochasticOscillator, ROCIndicator
#from ta.volatility import BollingerBands, AverageTrueRange

from scipy.stats import spearmanr
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import joblib
import os
import json
import logging
from pathlib import Path
from typing import Dict, Any, List, Tuple

import numpy as np
import pandas as pd

# ----------------------------
# Logging
# ----------------------------
logging.basicConfig(
    level=os.environ.get("LOG_LEVEL", "INFO"),
    format="%(asctime)s | %(levelname)s | %(message)s",
)
log = logging.getLogger("stage_06_window_scaling_seq2seq")

### 0.4. Definición de rutas

In [4]:
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

#### RUTAS DE ENTRADA

In [11]:
# ============================================================
# ENTRADA: T2 SPLITS
# ============================================================
IN_SUMMARY = DRIVE_DIR /  Path(os.environ.get("IN_PARQUET", "data/04_features/mnq_t2_summary.json"))
IN_SPLITS_SUMMARY = DRIVE_DIR / Path(os.environ.get("IN_SUMMARY", "data/05_splits/splits_summary.json"))
IN_PARQUET_T2_TRAIN = DRIVE_DIR / Path(os.environ.get("IN_PARQUET_T2_TRAIN", "data/05_splits/mnq_t2_train.parquet"))
IN_PARQUET_T2_VALID =  DRIVE_DIR / Path(os.environ.get("IN_PARQUET_T2_VALID", "data/05_splits/mnq_t2_valid.parquet"))
IN_PARQUET_T2_TEST =  DRIVE_DIR / Path(os.environ.get("IN_PARQUET_T2_TEST", "data/05_splits/mnq_t2_test.parquet"))




#### RUTAS DE SALIDA DE PARQUETS ESCALADOS

In [41]:
# ============================================================
# DATASETS ESCALADOS
# ============================================================
OUT_PARQUET_T2_TRAIN_Z = DRIVE_DIR / Path(os.environ.get("OUT_PARQUET_T2_TRAIN_Z", "data/06_scaled/mnq_t2_train_z.parquet"))
OUT_PARQUET_T2_VALID_Z =  DRIVE_DIR / Path(os.environ.get("OUT_PARQUET_T2_VALID_Z", "data/06_scaled/mnq_t2_valid_z.parquet"))
OUT_PARQUET_T2_TEST_Z  =  DRIVE_DIR / Path(os.environ.get("OUT_PARQUET_T2_TEST_Z",  "data/06_scaled/mnq_t2_test_z.parquet"))
OUT_SCALER_T2          =  DRIVE_DIR / Path(os.environ.get("OUT_SCALER_T2", "data/06_scaled/scaler_t2.pkl"))
OUT_SCALER_META_T2          =  DRIVE_DIR / Path(os.environ.get("OUT_SCALER_META_T2", "data/06_scaled/scaler_meta_t2.json"))

#### RUTAS DE SALIDA DE VENTANAS

In [13]:
from pathlib import Path
import os

# =========================
# SEQ2ONE (NPZ unificado por split)
# =========================

OUT_SEQ2ONE_T2_90_TRAIN = DRIVE_DIR / os.environ.get("OUT_SEQ2ONE_T2_90_TRAIN", "data/07_windows/seq2one/windows_t2_90_train.npz")
OUT_SEQ2ONE_T2_90_VALID = DRIVE_DIR / os.environ.get("OUT_SEQ2ONE_T2_90_VALID", "data/07_windows/seq2one/windows_t2_90_valid.npz")
OUT_SEQ2ONE_T2_90_TEST  = DRIVE_DIR / os.environ.get("OUT_SEQ2ONE_T2_90_TEST",  "data/07_windows/seq2one/windows_t2_90_test.npz")

OUT_SEQ2ONE_T2_120_TRAIN = DRIVE_DIR / os.environ.get("OUT_SEQ2ONE_T2_120_TRAIN", "data/07_windows/seq2one/windows_t2_120_train.npz")
OUT_SEQ2ONE_T2_120_VALID = DRIVE_DIR / os.environ.get("OUT_SEQ2ONE_T2_120_VALID", "data/07_windows/seq2one/windows_t2_120_valid.npz")
OUT_SEQ2ONE_T2_120_TEST  = DRIVE_DIR / os.environ.get("OUT_SEQ2ONE_T2_120_TEST",  "data/07_windows/seq2one/windows_t2_120_test.npz")



# **1. Carga de datos**

## 1.1. Carga de datasets `mnq_train`, `mnq_valid` y `mnq_test`






In [18]:
def _ensure_parent_dir(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)


def load_mnq_parquet(path: Path):

    if not os.path.exists(path):
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    print("Archivo encontrado en disco. Cargando dataset local...")

    mnq_parquet = pd.read_parquet(path)

    # ==========================================================
    # Orden temporal obligatorio
    # ==========================================================
    mnq_parquet = mnq_parquet.sort_values(
        by=["date", "minute_of_day"],
        ascending=[True, True]
    )

    # ==========================================================
    # Verificación
    # ==========================================================
    ordenado = mnq_parquet[["date", "minute_of_day"]].reset_index(drop=True).equals(
        mnq_parquet[["date", "minute_of_day"]]
        .sort_values(["date", "minute_of_day"])
        .reset_index(drop=True)
    )

    print("Orden temporal correcto:", ordenado)

    return mnq_parquet

## **1.2. Información de datasets**


In [17]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Optional, Tuple

import pandas as pd


def mnq_dataset_info(
    df: pd.DataFrame,
    *,
    name: str = "mnq_raw",
    tz_assume_if_naive: Optional[str] = None,  # ej: "UTC" o "America/New_York"
    day_def: str = "calendar",  # "calendar" (fecha calendario) o "trading" (días con datos)
) -> Dict[str, Any]:
    """
    Resume un dataset OHLCV con DatetimeIndex (ideal para mnq_raw).

    - Si el índice es tz-naive:
        - Si tz_assume_if_naive != None, lo localiza a esa tz.
        - Si no, reporta "tz-naive" (no se puede afirmar horario UTC).
    - Devuelve dict con métricas principales (y lo imprime bonito si se desea).
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError(f"{name}: se requiere DatetimeIndex, recibido: {type(df.index)}")

    idx = df.index

    # --- timezone / UTC info ---
    tzinfo = idx.tz
    if tzinfo is None:
        tz_status = "tz-naive (sin zona horaria)"
        if tz_assume_if_naive:
            idx = idx.tz_localize(tz_assume_if_naive)
            tzinfo = idx.tz
            tz_status = f"localizado como {tzinfo}"
    else:
        tz_status = f"{tzinfo}"

    # --- rango temporal ---
    ts_min = idx.min()
    ts_max = idx.max()

    first_day = ts_min.date()
    last_day = ts_max.date()

    # --- días ---
    if day_def == "calendar":
        total_days = (pd.Timestamp(last_day) - pd.Timestamp(first_day)).days + 1
    elif day_def == "trading":
        total_days = idx.normalize().nunique()
    else:
        raise ValueError("day_def debe ser 'calendar' o 'trading'")

    # --- columnas ---
    columns = list(df.columns)

    # --- checks útiles ---
    n_rows = len(df)
    n_cols = df.shape[1]
    n_missing = int(df.isna().sum().sum())
    missing_by_col = df.isna().sum().to_dict()
    dup_index = int(idx.duplicated().sum())
    is_monotonic = bool(idx.is_monotonic_increasing)

    # Frecuencia estimada (puede fallar si hay huecos grandes)
    freq = pd.infer_freq(idx[: min(50000, len(idx))])  # muestra grande pero acotada

    # Cobertura por día (min/max de hora del día, en tz del índice)
    # (Útil para ver si es 24/7 o horario de sesión)
    tod = pd.Series(idx.time)
    # Convertimos time a minutos del día para resumen robusto
    tod_minutes = pd.Series([t.hour * 60 + t.minute for t in tod])
    typical_minute_min = int(tod_minutes.min())
    typical_minute_max = int(tod_minutes.max())

    # Rango promedio de filas por día (sólo días con datos)
    rows_per_day = df.groupby(idx.normalize()).size()
    rows_per_day_stats = {
        "days_with_data": int(rows_per_day.shape[0]),
        "rows_per_day_min": int(rows_per_day.min()),
        "rows_per_day_p50": float(rows_per_day.median()),
        "rows_per_day_max": int(rows_per_day.max()),
    }

    # Si hay tz, también mostramos rango en UTC
    if tzinfo is not None:
        ts_min_utc = ts_min.tz_convert("UTC")
        ts_max_utc = ts_max.tz_convert("UTC")
        utc_range = (str(ts_min_utc), str(ts_max_utc))
        utc_note = "El índice está tz-aware; el horario UTC es inequívoco."
    else:
        utc_range = None
        utc_note = "El índice es tz-naive; no se puede asegurar si está en UTC sin suposiciones."

    info: Dict[str, Any] = {
        "name": name,
        "shape": (n_rows, n_cols),
        "columns": columns,
        "index_type": type(df.index).__name__,
        "index_tz": tz_status,
        "utc_note": utc_note,
        "datetime_min": str(ts_min),
        "datetime_max": str(ts_max),
        "first_day": str(first_day),
        "last_day": str(last_day),
        "total_days": int(total_days),
        "day_definition": day_def,
        "utc_range_if_applicable": utc_range,
        "is_index_monotonic_increasing": is_monotonic,
        "duplicated_timestamps_in_index": dup_index,
        "inferred_freq_sample": freq,
        "missing_total_cells": n_missing,
        "missing_by_col": missing_by_col,
        "rows_per_day_stats": rows_per_day_stats,
        "time_of_day_minutes_range": {
            "min_minute_of_day": typical_minute_min,
            "max_minute_of_day": typical_minute_max,
        },
    }
    return info


def print_mnq_dataset_info(info: Dict[str, Any]) -> None:
    """Imprime el dict de mnq_dataset_info de forma ordenada."""
    print(f"Dataset: {info['name']}")
    print(f"Shape: {info['shape']}")
    print(f"Columns: {info['columns']}")
    print(f"Index: {info['index_type']} | TZ: {info['index_tz']}")
    print(f"Datetime min/max: {info['datetime_min']}  ->  {info['datetime_max']}")
    print(f"First/Last day: {info['first_day']}  ->  {info['last_day']}")
    print(f"Total days ({info['day_definition']}): {info['total_days']}")
    #print(f"Inferred freq (sample): {info['inferred_freq_sample']}")
    #print(f"Index monotonic increasing: {info['is_index_monotonic_increasing']}")
    #print(f"Duplicated timestamps in index: {info['duplicated_timestamps_in_index']}")
    #print(f"Missing total cells: {info['missing_total_cells']}")
    #print(f"Missing by col: {info['missing_by_col']}")
    #print(f"Rows/day stats: {info['rows_per_day_stats']}")
    print(f"Time-of-day range (minutes): {info['time_of_day_minutes_range']}")
    print(f"UTC note: {info['utc_note']}")
    if info["utc_range_if_applicable"] is not None:
        print(f"UTC range: {info['utc_range_if_applicable'][0]}  ->  {info['utc_range_if_applicable'][1]}")


## **1.3. Carga de mnq e información**


In [19]:
# =========================
# T2
# =========================
mnq_t2_train = load_mnq_parquet(IN_PARQUET_T2_TRAIN)
info_mnq_t2_train = mnq_dataset_info(mnq_t2_train, name="mnq_t2_train", tz_assume_if_naive=None, day_def="trading")
print_mnq_dataset_info(info_mnq_t2_train)

mnq_t2_valid = load_mnq_parquet(IN_PARQUET_T2_VALID)
info_mnq_t2_valid = mnq_dataset_info(mnq_t2_valid, name="mnq_t2_valid", tz_assume_if_naive=None, day_def="trading")
print_mnq_dataset_info(info_mnq_t2_valid)

mnq_t2_test = load_mnq_parquet(IN_PARQUET_T2_TEST)
info_mnq_t2_test = mnq_dataset_info(mnq_t2_test, name="mnq_t2_test", tz_assume_if_naive=None, day_def="trading")
print_mnq_dataset_info(info_mnq_t2_test)


Archivo encontrado en disco. Cargando dataset local...
Orden temporal correcto: True
Dataset: mnq_t2_train
Shape: (462966, 18)
Columns: ['date', 'open', 'high', 'low', 'close', 'volume', 'minute_of_day', 'regime_id', 'roc_30', 'roc_60', 'stoch_k_30', 'atr_norm_10', 'close_fwd_90', 'delta_90', 'close_fwd_120', 'delta_120', 't2_dir_thr_90', 't2_dir_thr_120']
Index: DatetimeIndex | TZ: America/New_York
Datetime min/max: 2020-01-02 05:30:00-05:00  ->  2023-10-30 14:00:00-04:00
First/Last day: 2020-01-02  ->  2023-10-30
Total days (trading): 906
Time-of-day range (minutes): {'min_minute_of_day': 330, 'max_minute_of_day': 840}
UTC note: El índice está tz-aware; el horario UTC es inequívoco.
UTC range: 2020-01-02 10:30:00+00:00  ->  2023-10-30 18:00:00+00:00
Archivo encontrado en disco. Cargando dataset local...
Orden temporal correcto: True
Dataset: mnq_t2_valid
Shape: (99134, 18)
Columns: ['date', 'open', 'high', 'low', 'close', 'volume', 'minute_of_day', 'regime_id', 'roc_30', 'roc_60', 's

In [20]:
import json
import os
from pathlib import Path

# Cargar JSON
with open(IN_SUMMARY, "r") as f:
    summary = json.load(f)

# Extraer columnas
features_col = summary["column_groups"]["features"]
targets_col  = summary["column_groups"]["targets"]
aux_col      = summary["column_groups"]["auxiliary"]

# Check rápido
print("Features:", features_col)
print("Targets:", targets_col)
print("Aux:", aux_col)

Features: ['regime_id', 'roc_30', 'roc_60', 'stoch_k_30', 'atr_norm_10']
Targets: ['t2_dir_thr_90', 't2_dir_thr_120']
Aux: ['date', 'open', 'high', 'low', 'close', 'volume', 'minute_of_day', 'close_fwd_90', 'delta_90', 'close_fwd_120', 'delta_120']


In [22]:
# ==========================================================
# Diccionario con todos los datasets cargados
# ==========================================================
datasets = {
    "mnq_t2_train": mnq_t2_train,
    "mnq_t2_valid": mnq_t2_valid,
    "mnq_t2_test":  mnq_t2_test,
}

## **1.4. Verificación de orden temporal**

In [23]:
def comprobar_orden_datasets_base(**datasets):
    print("=" * 80)
    print("VALIDACIÓN ORDEN TEMPORAL - DATASETS BASE")
    print("=" * 80)

    for name, df in datasets.items():
        ordenado = df[["date", "minute_of_day"]].reset_index(drop=True).equals(
            df.sort_values(["date", "minute_of_day"])[["date", "minute_of_day"]].reset_index(drop=True)
        )

        print(f"{name}: {'OK' if ordenado else 'ERROR'}")

    print("=" * 80)

In [24]:
comprobar_orden_datasets_base(
    mnq_t2_train=mnq_t2_train,
    mnq_t2_valid=mnq_t2_valid,
    mnq_t2_test=mnq_t2_test,
)

VALIDACIÓN ORDEN TEMPORAL - DATASETS BASE
mnq_t2_train: OK
mnq_t2_valid: OK
mnq_t2_test: OK


# **2. Definición global de tamaños de ventana**

El `window_size` está condicionado por el feature que más historial necesita, en nuestro caso `roc_60`, necesitan 60 minutos previos para poder calcular su primer valor válido.

Si hacemos más corto el window_size corremos el riesgo de perder información o generar NaNs.


## **2.1. Implementación**

In [30]:
import pandas as pd
from typing import Optional, List, Dict, Any


# ============================================================
# Validaciones base para datasets intradía
# ============================================================

def validate_intraday_dataset_for_windows(
    df: pd.DataFrame,
    *,
    date_col: str = "date",
    minute_col: str = "minute_of_day",
    required_cols: Optional[List[str]] = None,
    check_nans_in: Optional[List[str]] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Valida que un dataset intradía esté en condiciones de usarse
    para construir ventanas temporales.

    Qué verifica
    -------------
    1. Que existan las columnas requeridas.
    2. Que el dataset esté ordenado por (date, minute_of_day).
    3. Que no existan duplicados por (date, minute_of_day).
    4. Opcionalmente, que no existan NaNs en ciertas columnas clave.

    Parámetros
    ----------
    df : pd.DataFrame
        Dataset a validar.
    date_col : str
        Columna que identifica la sesión/día.
    minute_col : str
        Columna de minuto intradía.
    required_cols : list[str] | None
        Columnas que deben existir obligatoriamente.
    check_nans_in : list[str] | None
        Columnas donde se desea verificar ausencia de NaNs.
    verbose : bool
        Si True, imprime el detalle de las verificaciones.

    Retorna
    -------
    dict
        Resumen de validación.
    """

    def _print(msg: str) -> None:
        if verbose:
            print(msg)

    _print("=" * 80)
    _print("VALIDACIÓN BASE DEL DATASET")
    _print("=" * 80)

    # --------------------------------------------------------
    # 1) Verificar columnas requeridas
    # --------------------------------------------------------
    required = [date_col, minute_col]
    if required_cols is not None:
        required += list(required_cols)

    missing_cols = [c for c in required if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Faltan columnas requeridas: {missing_cols}")

    _print(f"[OK] Columnas requeridas presentes: {required}")

    # --------------------------------------------------------
    # 2) Verificar orden por (date, minute_of_day)
    # --------------------------------------------------------
    check = df[[date_col, minute_col]].copy()
    check[date_col] = pd.to_datetime(check[date_col])

    check_sorted = check.sort_values([date_col, minute_col], kind="mergesort")

    is_sorted = check.reset_index(drop=True).equals(
        check_sorted.reset_index(drop=True)
    )

    if not is_sorted:
        raise ValueError(
            f"El dataset NO está ordenado por ({date_col}, {minute_col})"
        )

    _print(f"[OK] Orden correcto por ({date_col}, {minute_col})")

    # --------------------------------------------------------
    # 3) Verificar duplicados por timestamp intradía
    # --------------------------------------------------------
    n_duplicates = int(check.duplicated(subset=[date_col, minute_col]).sum())

    if n_duplicates > 0:
        raise ValueError(
            f"Se encontraron {n_duplicates} duplicados por ({date_col}, {minute_col})"
        )

    _print(f"[OK] Sin duplicados por ({date_col}, {minute_col})")

    # --------------------------------------------------------
    # 4) Verificar NaNs en columnas clave, si se solicita
    # --------------------------------------------------------
    nan_report = {}
    if check_nans_in is not None:
        for col in check_nans_in:
            if col not in df.columns:
                raise ValueError(f"La columna '{col}' no existe para chequeo de NaNs")

            n_nan = int(df[col].isna().sum())
            nan_report[col] = n_nan

            if n_nan > 0:
                raise ValueError(f"La columna '{col}' tiene {n_nan} NaNs")

        _print(f"[OK] Sin NaNs en columnas clave: {check_nans_in}")

    # --------------------------------------------------------
    # 5) Resumen
    # --------------------------------------------------------
    n_days = int(pd.to_datetime(df[date_col]).dt.date.nunique())

    _print(f"[INFO] Filas         : {len(df):,}")
    _print(f"[INFO] Días únicos   : {n_days:,}")
    _print("=" * 80)

    return {
        "n_rows": int(len(df)),
        "n_days": n_days,
        "is_sorted": True,
        "n_duplicates_timestamp": n_duplicates,
        "nan_report": nan_report,
    }


# ============================================================
# Regla de compatibilidad para ventanas
# ============================================================

def check_window_viability(
    df: pd.DataFrame,
    *,
    window_size: int,
    horizon: int,
    mode: str = "seq2one",
    date_col: str = "date",
    minute_col: str = "minute_of_day",
    required_cols: Optional[List[str]] = None,
    check_nans_in: Optional[List[str]] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Verifica si un tamaño de ventana es viable para un dataset dado.

    Lógica
    ------
    - seq2one:
        requiere al menos L observaciones por día
    - seq2seq:
        requiere al menos L + H observaciones por día

    Nota importante
    ---------------
    Para targets T2 ya precomputados en el dataset, en seq2one la
    restricción estructural correcta sigue siendo solo L.

    Parámetros
    ----------
    df : pd.DataFrame
        Dataset ya filtrado por split.
    window_size : int
        Lookback L.
    horizon : int
        Horizonte H del target.
    mode : str
        "seq2one" o "seq2seq".
    date_col : str
        Columna de sesión.
    minute_col : str
        Columna de minuto intradía.
    required_cols : list[str] | None
        Columnas extra requeridas para validar.
    check_nans_in : list[str] | None
        Columnas en las que se quiere exigir ausencia de NaNs.
    verbose : bool
        Si True, imprime verificaciones y resumen.

    Retorna
    -------
    dict
        Resumen de viabilidad de la ventana.
    """

    def _print(msg: str) -> None:
        if verbose:
            print(msg)

    if mode not in ["seq2one", "seq2seq"]:
        raise ValueError("mode debe ser 'seq2one' o 'seq2seq'")

    # --------------------------------------------------------
    # 1) Validación base del dataset antes del cálculo
    # --------------------------------------------------------
    base_validation = validate_intraday_dataset_for_windows(
        df,
        date_col=date_col,
        minute_col=minute_col,
        required_cols=required_cols,
        check_nans_in=check_nans_in,
        verbose=verbose,
    )

    # --------------------------------------------------------
    # 2) Longitud mínima requerida por día
    # --------------------------------------------------------
    required_length = window_size if mode == "seq2one" else window_size + horizon

    # Cantidad de filas por sesión
    counts_per_day = df.groupby(date_col).size().sort_index()

    # Días válidos e inválidos
    valid_days = counts_per_day[counts_per_day >= required_length]
    invalid_days = counts_per_day[counts_per_day < required_length]

    # Ventanas por día:
    # si un día tiene N observaciones y se requiere R,
    # entonces aporta N - R + 1 ventanas
    n_windows_per_day = (valid_days - required_length + 1).clip(lower=0)

    n_total_windows = int(n_windows_per_day.sum())

    # Algunas métricas útiles adicionales
    pct_valid_days = float(len(valid_days) / len(counts_per_day)) if len(counts_per_day) > 0 else 0.0
    avg_windows_per_valid_day = float(n_windows_per_day.mean()) if len(n_windows_per_day) > 0 else 0.0
    min_obs_day = int(counts_per_day.min()) if len(counts_per_day) > 0 else 0
    max_obs_day = int(counts_per_day.max()) if len(counts_per_day) > 0 else 0

    # --------------------------------------------------------
    # 3) Impresión de resultados
    # --------------------------------------------------------
    _print("=" * 80)
    _print("RESUMEN DE VIABILIDAD DE VENTANA")
    _print("=" * 80)
    _print(f"Modo                     : {mode}")
    _print(f"Window size (L)          : {window_size}")
    _print(f"Horizon (H)              : {horizon}")
    _print(f"Required length por día  : {required_length}")
    _print("-" * 80)
    _print(f"Días totales             : {len(counts_per_day)}")
    _print(f"Días válidos             : {len(valid_days)}")
    _print(f"Días descartados         : {len(invalid_days)}")
    _print(f"% días válidos           : {pct_valid_days:.2%}")
    _print("-" * 80)
    _print(f"Obs mín por día          : {min_obs_day}")
    _print(f"Obs máx por día          : {max_obs_day}")
    _print(f"Ventanas totales         : {n_total_windows}")
    _print(f"Prom ventanas/día válido : {avg_windows_per_valid_day:.2f}")
    _print("=" * 80)

    return {
        "window_size": window_size,
        "horizon": horizon,
        "mode": mode,
        "required_length": required_length,
        "n_rows": base_validation["n_rows"],
        "n_days_total": int(len(counts_per_day)),
        "n_days_valid": int(len(valid_days)),
        "n_days_invalid": int(len(invalid_days)),
        "pct_days_valid": pct_valid_days,
        "min_obs_day": min_obs_day,
        "max_obs_day": max_obs_day,
        "n_total_windows": n_total_windows,
        "avg_windows_per_valid_day": avg_windows_per_valid_day,
    }


# ============================================================
# Evaluación masiva para múltiples datasets y múltiples L
# ============================================================

def evaluate_window_grid(
    datasets: Dict[str, tuple],
    window_sizes: List[int],
    *,
    mode: str = "seq2one",
    date_col: str = "date",
    minute_col: str = "minute_of_day",
    required_cols: Optional[List[str]] = None,
    check_nans_in: Optional[List[str]] = None,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Evalúa una grilla de tamaños de ventana sobre múltiples datasets.

    Parámetros
    ----------
    datasets : dict
        Formato:
        {
            "nombre_dataset": (df, horizon),
            ...
        }
    window_sizes : list[int]
        Lista de ventanas L a evaluar.
    mode : str
        "seq2one" o "seq2seq".
    date_col : str
        Columna de sesión.
    minute_col : str
        Columna de minuto intradía.
    required_cols : list[str] | None
        Columnas extra requeridas.
    check_nans_in : list[str] | None
        Columnas a chequear sin NaNs.
    verbose : bool
        Si True, imprime detalle de cada corrida.

    Retorna
    -------
    pd.DataFrame
        Tabla consolidada con todos los resultados.
    """

    rows = []

    for dataset_name, (df, horizon) in datasets.items():

        if verbose:
            print("\n" + "=" * 90)
            print(f"DATASET: {dataset_name} | H={horizon}")
            print("=" * 90)

        for L in window_sizes:
            result = check_window_viability(
                df,
                window_size=L,
                horizon=horizon,
                mode=mode,
                date_col=date_col,
                minute_col=minute_col,
                required_cols=required_cols,
                check_nans_in=check_nans_in,
                verbose=verbose,
            )
            result["dataset"] = dataset_name
            rows.append(result)

    summary_df = pd.DataFrame(rows)

    # Reordenar columnas para lectura más clara
    preferred_cols = [
        "dataset",
        "mode",
        "window_size",
        "horizon",
        "required_length",
        "n_rows",
        "n_days_total",
        "n_days_valid",
        "n_days_invalid",
        "pct_days_valid",
        "min_obs_day",
        "max_obs_day",
        "n_total_windows",
        "avg_windows_per_valid_day",
    ]

    summary_df = summary_df[preferred_cols].sort_values(
        ["dataset", "window_size"]
    ).reset_index(drop=True)

    return summary_df

In [31]:
# ============================================================
# Definición global de tamaños de ventana
# ============================================================
window_sizes = [30, 60, 90, 120, 180]

# ============================================================
# Datasets a evaluar
# ============================================================
datasets = {
    "t2_90_train":  (mnq_t2_train, 90),
    "t2_90_valid":  (mnq_t2_valid, 90),
    "t2_90_test":   (mnq_t2_test, 90),
    "t2_120_train": (mnq_t2_train, 120),
    "t2_120_valid": (mnq_t2_valid, 120),
    "t2_120_test":  (mnq_t2_test, 120),
}

# ============================================================
# Columnas clave para validación
# Ajusta según el caso
# ============================================================
required_cols = ["date", "minute_of_day"]
check_nans_in = None
# Ejemplo si quieres exigir columnas concretas sin NaNs:
# check_nans_in = ["roc_30", "roc_60", "stoch_k_30", "atr_norm_10", "t2_dir_thr_90"]

# ============================================================
# Ejecutar evaluación consolidada
# ============================================================
window_viability_summary = evaluate_window_grid(
    datasets,
    window_sizes,
    mode="seq2one",
    date_col="date",
    minute_col="minute_of_day",
    required_cols=required_cols,
    check_nans_in=check_nans_in,
    verbose=True,
)

window_viability_summary


DATASET: t2_90_train | H=90
VALIDACIÓN BASE DEL DATASET
[OK] Columnas requeridas presentes: ['date', 'minute_of_day', 'date', 'minute_of_day']
[OK] Orden correcto por (date, minute_of_day)
[OK] Sin duplicados por (date, minute_of_day)
[INFO] Filas         : 462,966
[INFO] Días únicos   : 906
RESUMEN DE VIABILIDAD DE VENTANA
Modo                     : seq2one
Window size (L)          : 30
Horizon (H)              : 90
Required length por día  : 30
--------------------------------------------------------------------------------
Días totales             : 906
Días válidos             : 906
Días descartados         : 0
% días válidos           : 100.00%
--------------------------------------------------------------------------------
Obs mín por día          : 511
Obs máx por día          : 511
Ventanas totales         : 436692
Prom ventanas/día válido : 482.00
VALIDACIÓN BASE DEL DATASET
[OK] Columnas requeridas presentes: ['date', 'minute_of_day', 'date', 'minute_of_day']
[OK] Orden corr

,dataset,mode,window_size,horizon,required_length,n_rows,n_days_total,n_days_valid,n_days_invalid,pct_days_valid,min_obs_day,max_obs_day,n_total_windows,avg_windows_per_valid_day
0,t2_120_test,seq2one,30,120,30,99645,195,195,0,1.0,511,511,93990,482.0
1,t2_120_test,seq2one,60,120,60,99645,195,195,0,1.0,511,511,88140,452.0
2,t2_120_test,seq2one,90,120,90,99645,195,195,0,1.0,511,511,82290,422.0
3,t2_120_test,seq2one,120,120,120,99645,195,195,0,1.0,511,511,76440,392.0
4,t2_120_test,seq2one,180,120,180,99645,195,195,0,1.0,511,511,64740,332.0
5,t2_120_train,seq2one,30,120,30,462966,906,906,0,1.0,511,511,436692,482.0
6,t2_120_train,seq2one,60,120,60,462966,906,906,0,1.0,511,511,409512,452.0
7,t2_120_train,seq2one,90,120,90,462966,906,906,0,1.0,511,511,382332,422.0
8,t2_120_train,seq2one,120,120,120,462966,906,906,0,1.0,511,511,355152,392.0
9,t2_120_train,seq2one,180,120,180,462966,906,906,0,1.0,511,511,300792,332.0


## **2.2. Observaciones**

1. El dataset presenta una estructura intradía completamente consistente. Cada jornada contiene exactamente 511 observaciones, sin variabilidad en la longitud entre días. Además, no se identifican días inválidos ni truncados, lo que indica que el proceso de construcción y limpieza del dataset es correcto y no introduce pérdidas de información.

2. La generación de ventanas temporales es coherente con la teoría. Para cada tamaño de ventana (L), el número de muestras por día sigue exactamente la relación (N - L + 1), lo que confirma que no existen errores en la lógica de construcción de ventanas ni problemas de alineación temporal.

3. Los resultados son independientes del horizonte del target (H=90 y H=120), lo cual es consistente con el enfoque seq2one. Dado que los targets T2 ya incorporan el horizonte en su definición, la viabilidad de las ventanas depende únicamente del tamaño de la ventana (L), y no de (H).

4. El tamaño efectivo del dataset se reduce de forma controlada a medida que aumenta (L). Aunque ventanas más grandes generan menos muestras, incluso para (L=180) se dispone de un volumen significativo de datos (del orden de cientos de miles de observaciones), suficiente para entrenar modelos de manera robusta.

5. No existen restricciones estructurales para la elección del tamaño de ventana. Todos los valores evaluados son viables, sin pérdida de días ni degradación de la cobertura del dataset. Por lo tanto, la selección de (L) debe basarse exclusivamente en criterios de modelado, como capacidad predictiva, generalización y riesgo de sobreajuste.

6. La longitud fija por día constituye una propiedad altamente favorable del dataset. Permite comparabilidad directa entre muestras, evita problemas de padding o truncamiento, y simplifica tanto la construcción de ventanas como el entrenamiento de modelos secuenciales.

7. En conjunto, los resultados validan que el dataset y el pipeline de preparación están correctamente diseñados. No se detectan problemas de leakage, inconsistencias temporales ni limitaciones estructurales, por lo que se puede avanzar con confianza hacia la etapa de generación de ventanas y modelado.


## **2.3. Generación de summary de ventanas**

In [32]:
import json
from pathlib import Path


def generate_window_analysis_summary(
    window_viability_df,
    window_sizes,
    out_path: Path,
    verbose: bool = True,
):
    """
    Genera un summary del análisis de viabilidad de ventanas.

    Contenido:
    - grilla de ventanas utilizada
    - verificación estructural del dataset
    - métricas agregadas por ventana
    - conclusiones clave para modelado
    """

    def _print(msg):
        if verbose:
            print(msg)

    # ---------------------------------------------------
    # 1) Resumen estructural global
    # ---------------------------------------------------
    structure_summary = {
        "all_days_valid": bool((window_viability_df["n_days_invalid"] == 0).all()),
        "constant_intraday_length": bool(
            (window_viability_df["min_obs_day"] == window_viability_df["max_obs_day"]).all()
        ),
        "min_obs_per_day": int(window_viability_df["min_obs_day"].min()),
        "max_obs_per_day": int(window_viability_df["max_obs_day"].max()),
    }

    # ---------------------------------------------------
    # 2) Métricas por tamaño de ventana
    # ---------------------------------------------------
    window_stats = {}

    for L in window_sizes:
        subset = window_viability_df[window_viability_df["window_size"] == L]

        window_stats[str(L)] = {
            "n_total_windows_mean": int(subset["n_total_windows"].mean()),
            "avg_windows_per_day": float(subset["avg_windows_per_valid_day"].mean()),
            "pct_days_valid": float(subset["pct_days_valid"].mean()),
        }

    # ---------------------------------------------------
    # 3) Verificaciones clave
    # ---------------------------------------------------
    checks = {
        "no_invalid_days": structure_summary["all_days_valid"],
        "no_window_loss": structure_summary["all_days_valid"],
        "fixed_length_days": structure_summary["constant_intraday_length"],
        "consistent_window_generation": True,  # ya validado matemáticamente
    }

    # ---------------------------------------------------
    # 4) Conclusiones estructurales
    # ---------------------------------------------------
    conclusions = [
        "El dataset presenta longitud intradía constante en todas las sesiones.",
        "No se descartan días para ningún tamaño de ventana evaluado.",
        "La generación de ventanas es consistente con la relación N - L + 1.",
        "El horizonte del target no afecta la viabilidad en modo seq2one.",
        "La reducción de muestras al aumentar L es controlada y no limita el entrenamiento.",
        "Todos los tamaños de ventana son estructuralmente viables.",
        "La selección de L debe basarse en desempeño del modelo y no en restricciones del dataset.",
    ]

    # ---------------------------------------------------
    # 5) Objeto final
    # ---------------------------------------------------
    summary = {
        "window_grid": window_sizes,
        "structure_summary": structure_summary,
        "window_statistics": window_stats,
        "checks": checks,
        "conclusions": conclusions,
    }

    # ---------------------------------------------------
    # 6) Guardado
    # ---------------------------------------------------
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with open(out_path, "w") as f:
        json.dump(summary, f, indent=4)

    _print("=" * 100)
    _print("WINDOW ANALYSIS SUMMARY GENERADO")
    _print(f"Path: {out_path}")
    _print("=" * 100)

    return summary

In [33]:
OUT_WINDOW_SUMMARY = DRIVE_DIR / "data/07_windows/window_analysis_summary.json"

window_analysis_summary = generate_window_analysis_summary(
    window_viability_summary,
    window_sizes=window_sizes,
    out_path=OUT_WINDOW_SUMMARY,
)

WINDOW ANALYSIS SUMMARY GENERADO
Path: /content/drive/MyDrive/neural_profit/data/07_windows/window_analysis_summary.json


# **3. Escalado de los datasets**

## **3.1. Introducción teórica y principios de escalado**


El escalado constituye una etapa crítica dentro del pipeline de modelado, ya que la mayoría de los algoritmos de *machine learning* son sensibles a la escala de las variables de entrada. Diferencias en magnitud pueden provocar que ciertas features dominen el proceso de aprendizaje, afectando negativamente tanto la convergencia como el desempeño del modelo.

El dataset presenta heterogeneidad en las variables de entrada. Conviven indicadores técnicos, variables temporales y variables derivadas, cada una con rangos y distribuciones distintas. Esta diversidad requiere un tratamiento uniforme que evite sesgos inducidos por escala y permita una comparación adecuada entre variables.

El proceso de escalado debe respetar estrictamente la coherencia temporal del problema. En particular, el scaler se ajusta exclusivamente utilizando el conjunto de entrenamiento y luego se aplica sin modificación a los conjuntos de validación y prueba. Este procedimiento evita la introducción de información futura y garantiza la validez de la evaluación.

El escalado se aplica únicamente sobre variables continuas, incluyendo indicadores técnicos, variables derivadas y la variable temporal `minute_of_day`. No se transforman variables categóricas o discretas, como `regime_id`, ni columnas de identificación o fecha, ya que su significado no depende de la magnitud sino de su valor categórico o estructural.

El conjunto de features a escalar se define explícitamente y se mantiene consistente en todo el pipeline. Esto asegura reproducibilidad, trazabilidad y comparabilidad entre experimentos, especialmente al evaluar distintos modelos o configuraciones de ventana.

El escalado se realiza sobre los datos en formato tabular, antes de la generación de ventanas. Este orden permite conservar los nombres de las columnas, seleccionar de forma explícita las variables a transformar y evitar errores una vez que los datos son vectorizados.

El método de escalado debe ser adecuado para la naturaleza de los datos. En este caso, el uso de `StandardScaler` resulta apropiado, ya que centra las variables en media cero y varianza unitaria, facilitando el entrenamiento de modelos lineales, redes neuronales y algoritmos basados en distancia.

El escalado se integra como una transformación determinista dentro del pipeline. Esto implica guardar el scaler ajustado y reutilizarlo en todas las etapas posteriores, asegurando consistencia entre entrenamiento, validación, prueba e inferencia.

El objetivo del escalado no es alterar la información contenida en las features, sino proyectarlas a un espacio numérico comparable. Esto permite que el modelo aprenda relaciones estables entre variables, evita que ciertas features sean priorizadas artificialmente por su magnitud y mejora la capacidad de generalización entre distintos días y regímenes intradía.

Con estos criterios establecidos, el siguiente paso consiste en implementar el escalado de forma explícita y reproducible para los conjuntos `mnq_train`, `mnq_valid` y `mnq_test`.


## **3.2. Implementación de escalado**

### **3.2.1. Función para elegir escalador**


In [38]:
from __future__ import annotations

from typing import Optional
from sklearn.base import TransformerMixin
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler


def choose_scaler(
    scaler_type: str = "standard",
    *,
    # StandardScaler
    with_mean: bool = True,
    with_std: bool = True,
    # MinMaxScaler
    feature_range: tuple[float, float] = (0.0, 1.0),
    # RobustScaler
    quantile_range: tuple[float, float] = (25.0, 75.0),
    with_centering: bool = True,
    with_scaling: bool = True,
) -> Optional[TransformerMixin]:
    """
    Devuelve un scaler de sklearn según `scaler_type`.

    Opciones soportadas
    -------------------
    - 'standard' | 'z' | 'zscore' -> StandardScaler
    - 'minmax'   | 'min_max'      -> MinMaxScaler
    - 'robust'                    -> RobustScaler
    - 'none' | 'passthrough'      -> None

    Parámetros
    ----------
    scaler_type : str
        Tipo de escalador a utilizar.
    with_mean, with_std :
        Parámetros para StandardScaler.
    feature_range :
        Rango objetivo para MinMaxScaler.
    quantile_range, with_centering, with_scaling :
        Parámetros para RobustScaler.

    Retorna
    -------
    Optional[TransformerMixin]
        Instancia de scaler o None si no se desea escalado.
    """
    st = (scaler_type or "").strip().lower()

    if st in {"standard", "z", "zscore"}:
        return StandardScaler(with_mean=with_mean, with_std=with_std)

    if st in {"minmax", "min_max"}:
        return MinMaxScaler(feature_range=feature_range)

    if st == "robust":
        return RobustScaler(
            quantile_range=quantile_range,
            with_centering=with_centering,
            with_scaling=with_scaling,
        )

    if st in {"none", "passthrough"}:
        return None

    raise ValueError(
        f"scaler_type inválido: '{scaler_type}'. "
        "Use: 'standard', 'minmax', 'robust' o 'none'."
    )

### **3.2.2. Función para escalar datasets train, valid y test**


In [36]:
features_to_scale =[ 'roc_30', 'roc_60', 'stoch_k_30', 'atr_norm_10']

In [39]:
import pandas as pd
from typing import Tuple, Dict, Any, Sequence


def validar_orden_temporal(
    df: pd.DataFrame,
    *,
    name: str = "dataset",
    date_col: str = "date",
    minute_col: str = "minute_of_day",
) -> None:
    """
    Valida que el DataFrame esté ordenado temporalmente por:
    1) date ascendente
    2) minute_of_day ascendente dentro de cada date
    """
    if date_col not in df.columns:
        raise KeyError(f"{name}: no existe la columna '{date_col}'")

    if minute_col not in df.columns:
        raise KeyError(f"{name}: no existe la columna '{minute_col}'")

    ordenado = (
        df[[date_col, minute_col]]
        .reset_index(drop=True)
        .equals(
            df[[date_col, minute_col]]
            .sort_values([date_col, minute_col], kind="mergesort")
            .reset_index(drop=True)
        )
    )

    if not ordenado:
        raise ValueError(
            f"{name}: el dataset NO está ordenado por [{date_col}, {minute_col}]"
        )


def scale_mnq_splits(
    mnq_train: pd.DataFrame,
    mnq_valid: pd.DataFrame,
    mnq_test: pd.DataFrame,
    *,
    scaler,
    features_to_scale: Sequence[str],
    date_col: str = "date",
    minute_col: str = "minute_of_day",
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, Dict[str, Any]]:
    """
    Escala los splits evitando data leakage.

    Reglas
    ------
    - Valida orden temporal en train/valid/test.
    - Exige mismas columnas y mismo orden en los tres splits.
    - Fit del scaler SOLO en train.
    - Transform sobre train/valid/test usando las mismas columnas.
    - Escala únicamente las columnas de `features_to_scale`.
    """

    # ------------------------------------------------------------
    # 0. Validación de orden temporal
    # ------------------------------------------------------------
    validar_orden_temporal(
        mnq_train, name="mnq_train", date_col=date_col, minute_col=minute_col
    )
    validar_orden_temporal(
        mnq_valid, name="mnq_valid", date_col=date_col, minute_col=minute_col
    )
    validar_orden_temporal(
        mnq_test, name="mnq_test", date_col=date_col, minute_col=minute_col
    )

    # ------------------------------------------------------------
    # 1. Validación de columnas idénticas
    # ------------------------------------------------------------
    cols_train = list(mnq_train.columns)
    if list(mnq_valid.columns) != cols_train or list(mnq_test.columns) != cols_train:
        raise ValueError(
            "mnq_train, mnq_valid y mnq_test deben tener exactamente las mismas columnas y orden."
        )

    # ------------------------------------------------------------
    # 2. Validar que todas las columnas a escalar existan
    # ------------------------------------------------------------
    missing_scale_cols = [c for c in features_to_scale if c not in cols_train]
    if missing_scale_cols:
        raise ValueError(
            f"Faltan columnas esperadas en features_to_scale: {missing_scale_cols}"
        )

    scale_cols = list(features_to_scale)

    # ------------------------------------------------------------
    # 3. Validar NaNs en columnas a escalar
    # ------------------------------------------------------------
    for split_name, df_ in {
        "mnq_train": mnq_train,
        "mnq_valid": mnq_valid,
        "mnq_test": mnq_test,
    }.items():
        nan_counts = df_[scale_cols].isna().sum()
        nan_cols = nan_counts[nan_counts > 0]
        if len(nan_cols) > 0:
            raise ValueError(
                f"{split_name}: hay NaNs en columnas a escalar: {nan_cols.to_dict()}"
            )

    # ------------------------------------------------------------
    # 4. Si no hay scaler, devolver copias sin transformación
    # ------------------------------------------------------------
    if scaler is None:
        meta = {
            "scaled": False,
            "reason": "Scaler=None",
            "scale_cols": scale_cols,
            "temporal_order_validated": True,
        }
        return mnq_train.copy(), mnq_valid.copy(), mnq_test.copy(), meta

    tr = mnq_train.copy()
    va = mnq_valid.copy()
    te = mnq_test.copy()

    # ------------------------------------------------------------
    # 5. Fit SOLO en train
    # ------------------------------------------------------------
    scaler.fit(tr[scale_cols])

    # ------------------------------------------------------------
    # 6. Transform en train / valid / test
    # ------------------------------------------------------------
    tr.loc[:, scale_cols] = scaler.transform(tr[scale_cols])
    va.loc[:, scale_cols] = scaler.transform(va[scale_cols])
    te.loc[:, scale_cols] = scaler.transform(te[scale_cols])

    # ------------------------------------------------------------
    # 7. Metadata del escalado
    # ------------------------------------------------------------
    meta = {
        "scaled": True,
        "scaler_class": scaler.__class__.__name__,
        "scale_cols": scale_cols,
        "temporal_order_validated": True,
    }

    return tr, va, te, meta

In [ ]:
'''
features_to_scale = ['roc_30', 'roc_60', 'stoch_k_30', 'atr_norm_10']

scaler = choose_scaler("standard")

mnq_train_scaled, mnq_valid_scaled, mnq_test_scaled, scale_meta = scale_mnq_splits(
    mnq_train,
    mnq_valid,
    mnq_test,
    scaler=scaler,
    features_to_scale=features_to_scale,
    date_col="date",
    minute_col="minute_of_day",
)'''

### **3.2.3. Guardar dataset escalados**


In [40]:
import json
import joblib
import pandas as pd
from pathlib import Path
from typing import Tuple, Dict, Any, Sequence


def save_scaled_datasets(
    *,
    mnq_train_scaled: pd.DataFrame,
    mnq_valid_scaled: pd.DataFrame,
    mnq_test_scaled: pd.DataFrame,
    scale_meta: Dict[str, Any],
    out_train_path: Path,
    out_valid_path: Path,
    out_test_path: Path,
    out_meta_path: Path,
    scaler=None,
    out_scaler_path: Path | None = None,
) -> None:
    """
    Guarda datasets escalados, metadata y opcionalmente el scaler.

    Notas
    -----
    - Conserva el índice del DataFrame.
    - Compatible con arquitectura basada en rutas externas.
    """

    # ------------------------------------------------------------
    # 1) Crear directorios
    # ------------------------------------------------------------
    for path in [out_train_path, out_valid_path, out_test_path, out_meta_path]:
        path.parent.mkdir(parents=True, exist_ok=True)

    if scaler is not None and out_scaler_path is not None:
        out_scaler_path.parent.mkdir(parents=True, exist_ok=True)

    # ------------------------------------------------------------
    # 2) Guardar datasets escalados preservando índice
    # ------------------------------------------------------------
    mnq_train_scaled.to_parquet(out_train_path, index=True)
    mnq_valid_scaled.to_parquet(out_valid_path, index=True)
    mnq_test_scaled.to_parquet(out_test_path, index=True)

    # ------------------------------------------------------------
    # 3) Preparar metadata sin mutar el dict original
    # ------------------------------------------------------------
    meta_to_save = dict(scale_meta)
    meta_to_save["temporal_order_validated"] = True
    meta_to_save["sorted_by"] = ["date", "minute_of_day"]

    # ------------------------------------------------------------
    # 4) Guardar metadata
    # ------------------------------------------------------------
    with out_meta_path.open("w", encoding="utf-8") as f:
        json.dump(meta_to_save, f, indent=2, ensure_ascii=False)

    # ------------------------------------------------------------
    # 5) Guardar scaler si corresponde
    # ------------------------------------------------------------
    if scaler is not None and out_scaler_path is not None:
        joblib.dump(scaler, out_scaler_path)


def load_or_scale_mnq_datasets(
    *,
    mnq_train: pd.DataFrame,
    mnq_valid: pd.DataFrame,
    mnq_test: pd.DataFrame,
    scaler,
    features_to_scale: Sequence[str],
    out_train_path: Path,
    out_valid_path: Path,
    out_test_path: Path,
    out_meta_path: Path,
    out_scaler_path: Path,
    date_col: str = "date",
    minute_col: str = "minute_of_day",
    verbose: bool = True,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, Dict[str, Any], Any]:
    """
    Carga datasets escalados y scaler si ya existen.
    Si no existen, realiza el escalado (fit SOLO en train), guarda artefactos
    y retorna datasets escalados + metadata + scaler.

    Retorna
    -------
    mnq_train_s, mnq_valid_s, mnq_test_s, scale_meta, scaler
    """

    paths = {
        "train": out_train_path,
        "valid": out_valid_path,
        "test": out_test_path,
        "meta": out_meta_path,
        "scaler": out_scaler_path,
    }

    if verbose:
        print("Verificando existencia de datasets escalados y scaler...")
        for k, p in paths.items():
            print(f"  - {k}: {'OK' if p.exists() else 'NO EXISTE'}")

    files_exist = all(p.exists() for p in paths.values())

    # ------------------------------------------------------------
    # Caso 1: todo existe -> cargar
    # ------------------------------------------------------------
    if files_exist:
        if verbose:
            print("\nTodos los archivos existen. Cargando artefactos desde disco...")

        mnq_train_s = pd.read_parquet(out_train_path)
        mnq_valid_s = pd.read_parquet(out_valid_path)
        mnq_test_s = pd.read_parquet(out_test_path)

        with out_meta_path.open("r", encoding="utf-8") as f:
            scale_meta = json.load(f)

        scaler_loaded = joblib.load(out_scaler_path)

        if verbose:
            print("Carga completada. No se recalculó el escalado.")

        return mnq_train_s, mnq_valid_s, mnq_test_s, scale_meta, scaler_loaded

    # ------------------------------------------------------------
    # Caso 2: falta algún archivo -> recalcular
    # ------------------------------------------------------------
    if verbose:
        print("\nNo se encontraron todos los artefactos necesarios.")
        print("Recalculando escalado desde cero (fit SOLO en train)...")

    mnq_train_s, mnq_valid_s, mnq_test_s, scale_meta = scale_mnq_splits(
        mnq_train,
        mnq_valid,
        mnq_test,
        scaler=scaler,
        features_to_scale=features_to_scale,
        date_col=date_col,
        minute_col=minute_col,
    )

    save_scaled_datasets(
        mnq_train_scaled=mnq_train_s,
        mnq_valid_scaled=mnq_valid_s,
        mnq_test_scaled=mnq_test_s,
        scale_meta=scale_meta,
        out_train_path=out_train_path,
        out_valid_path=out_valid_path,
        out_test_path=out_test_path,
        out_meta_path=out_meta_path,
        scaler=scaler,
        out_scaler_path=out_scaler_path,
    )

    if verbose:
        print("Escalado completo y persistido en disco.")

    return mnq_train_s, mnq_valid_s, mnq_test_s, scale_meta, scaler


## **3.3. Aplicación**


In [45]:
features_to_scale = ["roc_30", "roc_60", "stoch_k_30", "atr_norm_10"]

scaler = choose_scaler("standard")

mnq_t2_train_s, mnq_t2_valid_s, mnq_t2_test_s, scale_t2_meta, scaler_t2 = load_or_scale_mnq_datasets(
    mnq_train=mnq_t2_train,
    mnq_valid=mnq_t2_valid,
    mnq_test=mnq_t2_test,
    scaler=scaler,
    features_to_scale=features_to_scale,
    out_train_path=OUT_PARQUET_T2_TRAIN_Z,
    out_valid_path=OUT_PARQUET_T2_VALID_Z,
    out_test_path=OUT_PARQUET_T2_TEST_Z,
    out_meta_path=OUT_SCALER_META_T2,
    out_scaler_path=OUT_SCALER_T2,
    date_col="date",
    minute_col="minute_of_day",
    verbose=True,
)

Verificando existencia de datasets escalados y scaler...
  - train: OK
  - valid: OK
  - test: OK
  - meta: OK
  - scaler: OK

Todos los archivos existen. Cargando artefactos desde disco...
Carga completada. No se recalculó el escalado.


## **3.4. Verificación de orden temporal**


In [44]:
def verificar_orden_dataset(
    target_name: str,
    df: pd.DataFrame,
    meta: dict,
    date_col: str = "date",
    minute_col: str = "minute_of_day",
):
    """
    Verifica que el dataset esté ordenado temporalmente y que la metadata
    registre la validación temporal.
    """

    print(f"\n=== Verificación temporal: {target_name} ===")

    # -------------------------------------------------------
    # 1. Verificar metadata
    # -------------------------------------------------------
    meta_flag = meta.get("temporal_order_validated", False)

    print("Metadata temporal_order_validated:", meta_flag)

    # -------------------------------------------------------
    # 2. Verificar orden real del dataframe
    # -------------------------------------------------------
    ordered = (
        df[[date_col, minute_col]]
        .reset_index(drop=True)
        .equals(
            df[[date_col, minute_col]]
            .sort_values([date_col, minute_col])
            .reset_index(drop=True)
        )
    )

    print("Dataset ordenado temporalmente:", ordered)

    # -------------------------------------------------------
    # 3. Resultado final
    # -------------------------------------------------------
    if meta_flag and ordered:
        print("✔ Dataset temporalmente consistente")
    else:
        print("✘ Posible problema de orden temporal")

    return meta_flag, ordered

In [47]:
verificar_orden_dataset(
    "mnq_t2_s",
    mnq_t2_train_s,
    scale_t2_meta,
)



=== Verificación temporal: mnq_t2_s ===
Metadata temporal_order_validated: True
Dataset ordenado temporalmente: True
✔ Dataset temporalmente consistente


(True, True)

## **3.5. Verificación de escalado**


In [53]:
import pandas as pd
import numpy as np


def verify_scaling(
    train_raw: pd.DataFrame,
    train_scaled: pd.DataFrame,
    valid_raw: pd.DataFrame,
    valid_scaled: pd.DataFrame,
    test_raw: pd.DataFrame,
    test_scaled: pd.DataFrame,
    *,
    name: str,
    target_col: str,
    features_to_scale,
    date_col: str = "date",
    minute_col: str = "minute_of_day",
    tol_mean: float = 0.05,
    tol_std: float = 0.05,
):
    """
    Verifica que el escalado se haya realizado correctamente.

    Chequeos:
    - TRAIN: media ~ 0 y std ~ 1
    - VALID/TEST: fueron transformados (≠ raw)
    - Target no fue modificado
    - Columnas de fecha no fueron modificadas
    - Orden temporal preservado
    - Índice (DatetimeIndex) preservado
    """

    print("\n" + "=" * 20 + f" {name} " + "=" * 20)

    # ------------------------------------------------------------
    # 1) Features presentes
    # ------------------------------------------------------------
    scale_cols = [c for c in features_to_scale if c in train_raw.columns]
    print(f"Features a verificar (presentes): {len(scale_cols)}")

    # ------------------------------------------------------------
    # 2) TRAIN: verificar estandarización
    # ------------------------------------------------------------
    means = train_scaled[scale_cols].mean()
    stds = train_scaled[scale_cols].std(ddof=0)

    mean_fail = (means.abs() > tol_mean).sum()
    std_fail = ((stds - 1).abs() > tol_std).sum()

    print("\nTRAIN (estandarización):")
    print(f"  Mean fuera tolerancia (|mean| > {tol_mean}): {mean_fail}")
    print(f"  Std  fuera tolerancia (|std-1| > {tol_std}): {std_fail}")

    # ------------------------------------------------------------
    # 3) VALID / TEST: verificar transformación
    # ------------------------------------------------------------
    def _is_transformed(raw, scaled):
        return not np.allclose(raw[scale_cols].values, scaled[scale_cols].values)

    print("\nTransformación aplicada:")
    print(f"  VALID transformado: {_is_transformed(valid_raw, valid_scaled)}")
    print(f"  TEST  transformado: {_is_transformed(test_raw, test_scaled)}")

    # ------------------------------------------------------------
    # 4) Target sin modificar
    # ------------------------------------------------------------
    def _same_col(a, b, col):
        return np.array_equal(a[col].values, b[col].values)

    print("\nTarget sin modificar:")
    print(f"  TRAIN: {_same_col(train_raw, train_scaled, target_col)}")
    print(f"  VALID: {_same_col(valid_raw, valid_scaled, target_col)}")
    print(f"  TEST : {_same_col(test_raw, test_scaled, target_col)}")

    # ------------------------------------------------------------
    # 5) Date sin modificar
    # ------------------------------------------------------------
    print("\nDate sin modificar:")
    print(f"  TRAIN: {_same_col(train_raw, train_scaled, date_col)}")
    print(f"  VALID: {_same_col(valid_raw, valid_scaled, date_col)}")
    print(f"  TEST : {_same_col(test_raw, test_scaled, date_col)}")

    # ------------------------------------------------------------
    # 6) Secuencia temporal preservada
    # ------------------------------------------------------------
    def _same_time_order(a, b):
        return a[[date_col, minute_col]].reset_index(drop=True).equals(
            b[[date_col, minute_col]].reset_index(drop=True)
        )

    print("\nMisma secuencia temporal que raw:")
    print(f"  TRAIN: {_same_time_order(train_raw, train_scaled)}")
    print(f"  VALID: {_same_time_order(valid_raw, valid_scaled)}")
    print(f"  TEST : {_same_time_order(test_raw, test_scaled)}")

    # ------------------------------------------------------------
    # 7) Orden temporal del dataset escalado
    # ------------------------------------------------------------
    def _is_sorted(df):
        return df[[date_col, minute_col]].reset_index(drop=True).equals(
            df[[date_col, minute_col]]
            .sort_values([date_col, minute_col])
            .reset_index(drop=True)
        )

    print("\nScaled sigue ordenado temporalmente:")
    print(f"  TRAIN: {_is_sorted(train_scaled)}")
    print(f"  VALID: {_is_sorted(valid_scaled)}")
    print(f"  TEST : {_is_sorted(test_scaled)}")

    # ------------------------------------------------------------
    # 8) Índice (DatetimeIndex) preservado
    # ------------------------------------------------------------
    def _check_index_equal(df_raw, df_scaled):
        if isinstance(df_raw.index, pd.DatetimeIndex) and isinstance(df_scaled.index, pd.DatetimeIndex):
            return df_raw.index.equals(df_scaled.index)
        return None

    idx_train = _check_index_equal(train_raw, train_scaled)
    idx_valid = _check_index_equal(valid_raw, valid_scaled)
    idx_test  = _check_index_equal(test_raw,  test_scaled)

    print("\nÍndice (DatetimeIndex) sin modificar:")
    print(f"  TRAIN: {idx_train if idx_train is not None else 'N/A'}")
    print(f"  VALID: {idx_valid if idx_valid is not None else 'N/A'}")
    print(f"  TEST : {idx_test  if idx_test  is not None else 'N/A'}")

In [50]:
targets_col

['t2_dir_thr_90', 't2_dir_thr_120']

In [54]:
# =========================
# t2_dir_thr_90
# =========================
verify_scaling(
    mnq_t2_train, mnq_t2_train_s,
    mnq_t2_valid, mnq_t2_valid_s,
    mnq_t2_test,  mnq_t2_test_s,
    name=f"mnq_t2 - {targets_col[0]}",
    target_col=targets_col[0],
    features_to_scale=features_to_scale,
)

# =========================
# t2_dir_thr_120
# =========================
verify_scaling(
    mnq_t2_train, mnq_t2_train_s,
    mnq_t2_valid, mnq_t2_valid_s,
    mnq_t2_test,  mnq_t2_test_s,
    name=f"mnq_t2 - {targets_col[1]}",
    target_col=targets_col[1],
    features_to_scale=features_to_scale,
)




==================== mnq_t2 - t2_dir_thr_90 ====================
Features a verificar (presentes): 4

TRAIN (estandarización):
  Mean fuera tolerancia (|mean| > 0.05): 0
  Std  fuera tolerancia (|std-1| > 0.05): 0

Transformación aplicada:
  VALID transformado: True
  TEST  transformado: True

Target sin modificar:
  TRAIN: True
  VALID: True
  TEST : True

Date sin modificar:
  TRAIN: True
  VALID: True
  TEST : True

Misma secuencia temporal que raw:
  TRAIN: True
  VALID: True
  TEST : True

Scaled sigue ordenado temporalmente:
  TRAIN: True
  VALID: True
  TEST : True

Índice (DatetimeIndex) sin modificar:
  TRAIN: True
  VALID: True
  TEST : True

==================== mnq_t2 - t2_dir_thr_120 ====================
Features a verificar (presentes): 4

TRAIN (estandarización):
  Mean fuera tolerancia (|mean| > 0.05): 0
  Std  fuera tolerancia (|std-1| > 0.05): 0

Transformación aplicada:
  VALID transformado: True
  TEST  transformado: True

Target sin modificar:
  TRAIN: True
  VALI

1. El escalado en `TRAIN` es correcto. Todas las features presentan media cercana a 0 y desviación estándar cercana a 1, sin desviaciones fuera de tolerancia.

2. La transformación se aplicó correctamente en `VALID` y `TEST`, lo que confirma que el scaler fue utilizado de forma consistente fuera del entrenamiento.

3. El target no fue modificado en ningún split, garantizando que no hubo contaminación durante el proceso de escalado.

4. La columna `date` se mantiene intacta, preservando la trazabilidad temporal del dataset.

5. La secuencia temporal se conserva exactamente igual entre los datasets raw y scaled, lo que indica que no hubo reordenamientos.

6. Los datasets escalados siguen correctamente ordenados en el tiempo, lo cual es crítico para las etapas posteriores.

7. El `DatetimeIndex` se preserva sin cambios en todos los splits, confirmando que no se perdió la estructura temporal al guardar o cargar los datos.

8. Los resultados son idénticos para ambos targets (`t2_dir_thr_90` y `t2_dir_thr_120`), lo cual es esperable ya que el escalado depende solo de las features.

9. En conjunto, el proceso de escalado es correcto, consistente y libre de data leakage, por lo que el pipeline puede avanzar a la siguiente etapa sin ajustes.


# **4. Generación de ventanas deslizantes (sliding windows)**

## **4.1. Introducción conceptual**

La generación de ventanas constituye una etapa fundamental en la preparación de los datos para el modelado, ya que permite transformar el dataset tabular en una representación secuencial adecuada para capturar dependencias temporales en el comportamiento del mercado.

En este proyecto se adopta un enfoque **seq2one**, en el cual cada muestra está compuesta por una secuencia de observaciones pasadas (ventana) y un único valor objetivo asociado.

Cada ventana de entrada $X_i$ está formada por una secuencia de longitud fija $L$, construida a partir de las features en filas consecutivas del dataset:

$$
X_i = [x_i, x_{i+1}, ..., x_{i+L-1}]
$$

La salida asociada $ y_i $ corresponde al valor del target en la **última fila de la ventana**:

$$
y_i = y_{i+L-1}
$$

En este caso, los targets utilizados son:

* `t2_dir_thr_90`
* `t2_dir_thr_120`

Estos targets ya han sido construidos previamente incorporando un horizonte hacia adelante, por lo que no es necesario aplicar desplazamientos adicionales durante la generación de ventanas. Como consecuencia, la relación entre las ventanas y el target es directa y no depende explícitamente del horizonte en esta etapa.

Dado un día con $ N $ observaciones, el número de ventanas generadas es:

$$
N - L + 1
$$

Este esquema garantiza que cada ventana utiliza únicamente información pasada para predecir el target asociado, respetando la causalidad temporal del problema.

La generación de ventanas se realiza de forma independiente por cada jornada intradía, evitando la mezcla de información entre días distintos. Esto es crítico para preservar la estructura del mercado y evitar cualquier forma de *data leakage* entre sesiones.

Cada muestra generada tiene la siguiente estructura:

* Entrada: matriz de dimensión $ L \times F $, donde  $F$ es el número de features.
* Salida: un valor escalar correspondiente al target.

El tamaño de la ventana $ L $ es un hiperparámetro clave que controla la cantidad de información histórica disponible para el modelo. En este proyecto se utiliza una grilla fija de valores para $ L $, con el objetivo de garantizar comparabilidad entre experimentos.

Finalmente, la generación de ventanas se realiza sobre los datasets ya escalados, asegurando que todas las features se encuentren en una escala homogénea antes de ser utilizadas por los modelos.

## **4.2. Construcción de ventanas `seq2one`**

### **4.2.1. Generador seq2one**

In [85]:
import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view
from typing import List, Tuple

import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view
from typing import List, Tuple


def generate_windows_seq2one(
    df: pd.DataFrame,
    *,
    date_col: str,
    features: List[str],
    target_col: str,
    window_size: int,
    minute_col: str = "minute_of_day",
    flatten: bool = False,
    drop_windows_with_nan: bool = True,
    validate_temporal_order: bool = True,
    validate_unique_minutes: bool = True,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Genera ventanas seq2one a partir de un DataFrame intradía.

    Esquema
    -------
    Para cada día:
      - X_i = [x_i, ..., x_{i+L-1}]
      - y_i = target en la última fila de la ventana

    Esto implica:
      y_i = y[i + window_size - 1]

    Parámetros
    ----------
    df : pd.DataFrame
        Dataset intradía con múltiples sesiones.
    date_col : str
        Columna que identifica la sesión/día.
    features : List[str]
        Features de entrada.
    target_col : str
        Columna target.
    window_size : int
        Longitud L de la ventana.
    minute_col : str
        Columna de orden intradía.
    flatten : bool
        Si True, retorna X con shape (N, L*F).
    drop_windows_with_nan : bool
        Si True, descarta ventanas con NaNs en X o y.
    validate_temporal_order : bool
        Si True, exige orden ascendente por minuto dentro de cada día.
    validate_unique_minutes : bool
        Si True, exige unicidad de minute_col dentro de cada día.

    Retorna
    -------
    X : np.ndarray
        - flatten=False -> shape (N, L, F), dtype float32
        - flatten=True  -> shape (N, L*F), dtype float32
    y : np.ndarray
        shape (N,), dtype entero para targets T2 si corresponde
    """

    # ------------------------------------------------------------
    # 1) Validaciones básicas
    # ------------------------------------------------------------
    if window_size <= 0:
        raise ValueError("window_size debe ser un entero positivo.")

    required_cols = [date_col] + features + [target_col]
    if validate_temporal_order or validate_unique_minutes:
        required_cols.append(minute_col)

    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Faltan columnas requeridas en df: {missing}")

    # ------------------------------------------------------------
    # 2) Acumuladores
    # ------------------------------------------------------------
    X_all: List[np.ndarray] = []
    y_all: List[np.ndarray] = []
    F = len(features)

    # ------------------------------------------------------------
    # 3) Generación por día
    # ------------------------------------------------------------
    for day_value, g in df.groupby(date_col, sort=False):
        g = g.reset_index(drop=True)
        n = len(g)

        # --------------------------------------------------------
        # 3.1) Validaciones dentro del día
        # --------------------------------------------------------
        if validate_temporal_order and not g[minute_col].is_monotonic_increasing:
            raise ValueError(
                f"Desorden temporal detectado en {date_col}={day_value}: "
                f"'{minute_col}' no está en orden ascendente."
            )

        if validate_unique_minutes:
            duplicated_mask = g[minute_col].duplicated(keep=False)
            if duplicated_mask.any():
                duplicated_values = g.loc[duplicated_mask, minute_col].tolist()[:10]
                raise ValueError(
                    f"Minutos duplicados detectados en {date_col}={day_value}: "
                    f"{duplicated_values}"
                )

        # Si no alcanza para una ventana completa
        if n < window_size:
            continue

        # --------------------------------------------------------
        # 3.2) Conversión a numpy
        # --------------------------------------------------------
        Xg = g[features].to_numpy(dtype=np.float32, copy=False)

        # Preservar dtype original del target al inicio
        yg = g[target_col].to_numpy(copy=False)

        # --------------------------------------------------------
        # 3.3) Ventanas deslizantes sobre X
        # --------------------------------------------------------
        Xw = sliding_window_view(Xg, window_shape=window_size, axis=0)

        # Normalizar a shape (N, L, F)
        if Xw.ndim != 3:
            raise ValueError(f"Xw ndim inesperado: {Xw.ndim} | shape={Xw.shape}")

        if Xw.shape[1] == F and Xw.shape[2] == window_size:
            Xw = np.swapaxes(Xw, 1, 2)

        if Xw.shape[1] != window_size or Xw.shape[2] != F:
            raise ValueError(
                f"Xw shape inválido tras normalizar: {Xw.shape}. "
                f"Esperado: (N, {window_size}, {F})"
            )

        # --------------------------------------------------------
        # 3.4) Target alineado al final de la ventana
        # --------------------------------------------------------
        yw = yg[window_size - 1:]

        # --------------------------------------------------------
        # 3.5) Filtrado de NaNs
        # --------------------------------------------------------
        if drop_windows_with_nan:
            x_ok = ~np.isnan(Xw).any(axis=(1, 2))
            y_ok = ~pd.isna(yw)

            ok = x_ok & y_ok
            Xw = Xw[ok]
            yw = yw[ok]

        if Xw.size == 0:
            continue

        # --------------------------------------------------------
        # 3.6) Aplanado opcional
        # --------------------------------------------------------
        if flatten:
            Xw = Xw.reshape(Xw.shape[0], -1)

        # --------------------------------------------------------
        # 3.7) Cast final
        # --------------------------------------------------------
        Xw = Xw.astype(np.float32, copy=False)

        # Para targets T2: si vienen como float pero representan clases,
        # convertirlos explícitamente a entero
        if np.issubdtype(np.asarray(yw).dtype, np.floating):
            yw = np.asarray(yw, dtype=np.int8)
        else:
            yw = np.asarray(yw)

        X_all.append(Xw)
        y_all.append(yw)

    # ------------------------------------------------------------
    # 4) Caso borde: no se generaron ventanas
    # ------------------------------------------------------------
    if not X_all:
        X = np.empty((0, window_size, F), dtype=np.float32)
        if flatten:
            X = X.reshape(0, window_size * F)

        y = np.empty((0,), dtype=np.int8)
        return X, y

    # ------------------------------------------------------------
    # 5) Concatenación final
    # ------------------------------------------------------------
    X = np.concatenate(X_all, axis=0)
    y = np.concatenate(y_all, axis=0)

    return X, y

### **4.2.2. Load/Build para SEQ2ONE**

In [56]:
import json
import numpy as np
from pathlib import Path
from typing import List, Tuple


def prepare_or_load_seq2one_windows_npz(
    *,
    mnq_train,
    mnq_valid,
    mnq_test,
    features: List[str],
    target_col: str,
    window_size: int,
    out_train,
    out_valid,
    out_test,
    date_col: str = "date",
    minute_col: str = "minute_of_day",
    flatten: bool = False,
    drop_windows_with_nan: bool = True,
    verbose: bool = True,
    repair_axis_order_if_needed: bool = True,
    validate_temporal_order: bool = True,
    validate_unique_minutes: bool = True,
    out_meta_train=None,
    out_meta_valid=None,
    out_meta_test=None,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Prepara o carga ventanas seq2one desde disco en formato .npz.

    Flujo
    -----
    Para cada split:
      - Si existe el .npz, lo carga.
      - Si no existe, construye ventanas con generate_windows_seq2one(...) y las guarda.
      - Opcionalmente repara el orden de ejes si detecta (N, F, L).
      - Opcionalmente guarda metadata por split.

    Retorna
    -------
    (X_train, y_train, X_valid, y_valid, X_test, y_test)
    """

    # ------------------------------------------------------------
    # 1) Helpers de path
    # ------------------------------------------------------------
    def _to_path(p) -> Path | None:
        if p is None:
            return None
        return p if isinstance(p, Path) else Path(str(p))

    def _ensure_parent_dir(path: Path) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)

    out_train = _to_path(out_train)
    out_valid = _to_path(out_valid)
    out_test = _to_path(out_test)

    out_meta_train = _to_path(out_meta_train)
    out_meta_valid = _to_path(out_meta_valid)
    out_meta_test = _to_path(out_meta_test)

    if out_train is None or out_valid is None or out_test is None:
        raise ValueError("out_train, out_valid y out_test son obligatorios.")

    F = len(features)

    # ------------------------------------------------------------
    # 2) Helper para verificar orden temporal del DataFrame
    # ------------------------------------------------------------
    def _check_temporal_order(df, split_name: str) -> None:
        if not validate_temporal_order:
            return

        required = [date_col, minute_col]
        missing = [c for c in required if c not in df.columns]
        if missing:
            raise ValueError(
                f"[{split_name}] faltan columnas para validar orden temporal: {missing}"
            )

        ordered = (
            df[[date_col, minute_col]]
            .reset_index(drop=True)
            .equals(
                df[[date_col, minute_col]]
                .sort_values([date_col, minute_col], kind="mergesort")
                .reset_index(drop=True)
            )
        )

        if not ordered:
            raise ValueError(
                f"[{split_name}] el DataFrame no está ordenado por "
                f"[{date_col}, {minute_col}] antes de generar ventanas."
            )

    # ------------------------------------------------------------
    # 3) Reparación opcional del orden de ejes en X
    # ------------------------------------------------------------
    def _maybe_repair_X(X: np.ndarray) -> np.ndarray:
        if flatten or not repair_axis_order_if_needed:
            return X

        if X.ndim == 3 and X.shape[1] == F and X.shape[2] == window_size:
            return np.swapaxes(X, 1, 2)  # (N,F,L) -> (N,L,F)

        return X

    # ------------------------------------------------------------
    # 4) Guardado / carga NPZ
    # ------------------------------------------------------------
    def _save_npz(path_npz: Path, X: np.ndarray, y: np.ndarray) -> None:
        _ensure_parent_dir(path_npz)

        # X en float32 para preservar estabilidad numérica
        X_to_save = np.asarray(X, dtype=np.float32)

        # y preserva su naturaleza (por ejemplo, entero para targets T2)
        y_to_save = np.asarray(y)

        np.savez_compressed(path_npz, X=X_to_save, y=y_to_save)

    def _load_npz(path_npz: Path):
        with np.load(path_npz, allow_pickle=False) as z:
            X = z["X"]
            y = z["y"]
        return X, y

    # ------------------------------------------------------------
    # 5) Guardado de metadata opcional
    # ------------------------------------------------------------
    def _save_meta(path_meta: Path | None, split_name: str, X: np.ndarray, y: np.ndarray) -> None:
        if path_meta is None:
            return

        _ensure_parent_dir(path_meta)

        meta = {
            "split": split_name,
            "target_col": target_col,
            "features": list(features),
            "n_features": len(features),
            "window_size": window_size,
            "flatten": flatten,
            "drop_windows_with_nan": drop_windows_with_nan,
            "validate_temporal_order": validate_temporal_order,
            "validate_unique_minutes": validate_unique_minutes,
            "date_col": date_col,
            "minute_col": minute_col,
            "n_samples": int(len(X)),
            "X_shape": list(X.shape),
            "y_shape": list(y.shape),
            "X_dtype": str(X.dtype),
            "y_dtype": str(y.dtype),
            "sorted_by": [date_col, minute_col],
        }

        with path_meta.open("w", encoding="utf-8") as f:
            json.dump(meta, f, indent=2, ensure_ascii=False)

    # ------------------------------------------------------------
    # 6) Lógica principal por split
    # ------------------------------------------------------------
    def _load_or_build(split_name: str, df, path_npz: Path, path_meta: Path | None):
        action = "load"

        if not path_npz.exists():
            action = "build"

            _check_temporal_order(df, split_name)

            X, y = generate_windows_seq2one(
                df=df,
                date_col=date_col,
                features=features,
                target_col=target_col,
                window_size=window_size,
                minute_col=minute_col,
                flatten=flatten,
                drop_windows_with_nan=drop_windows_with_nan,
                validate_temporal_order=validate_temporal_order,
                validate_unique_minutes=validate_unique_minutes,
            )

            _save_npz(path_npz, X, y)
            _save_meta(path_meta, split_name, X, y)

        else:
            X, y = _load_npz(path_npz)

        # --------------------------------------------------------
        # Reparación opcional de ejes
        # --------------------------------------------------------
        X2 = _maybe_repair_X(X)

        if (X2 is not X) and action == "load":
            X2_np = np.asarray(X2, dtype=np.float32)
            y_np = np.asarray(y)
            _save_npz(path_npz, X2_np, y_np)
            _save_meta(path_meta, split_name, X2_np, y_np)
            X = X2_np
            y = y_np
        else:
            X = X2

        if verbose:
            print(
                f"[{target_col} | L={window_size} | {split_name}] {action.upper()}  "
                f"X={X.shape}  y={y.shape}  -> {path_npz.name}"
            )

        return X, y

    # ------------------------------------------------------------
    # 7) Ejecutar para train / valid / test
    # ------------------------------------------------------------
    X_train, y_train = _load_or_build("train", mnq_train, out_train, out_meta_train)
    X_valid, y_valid = _load_or_build("valid", mnq_valid, out_valid, out_meta_valid)
    X_test, y_test = _load_or_build("test", mnq_test, out_test, out_meta_test)

    return X_train, y_train, X_valid, y_valid, X_test, y_test

### **4.2.3. Info para SEQ2ONE**

In [64]:
from pathlib import Path
import numpy as np
import pandas as pd


def xy_info_seq2one_compact(
    target_col: str,
    X_train, y_train,
    X_valid, y_valid,
    X_test,  y_test,
    *,
    window_size: int,
    n_features: int,
    flatten: bool = False,
    path_train: str | Path | None = None,
    path_valid: str | Path | None = None,
    path_test:  str | Path | None = None,
):
    """
    Muestra información estructural compacta de ventanas seq2one por split.

    Reporta:
    - shape de X e y
    - dtype de X e y
    - validación de dimensiones esperadas
    - presencia de NaN
    - tamaño estimado en RAM
    - tamaño en disco del .npz (si se provee path)
    - clases únicas de y (útil para targets de clasificación)
    """

    # ------------------------------------------------------------
    # 1) Helpers
    # ------------------------------------------------------------
    def _to_path(p):
        return None if p is None else (p if isinstance(p, Path) else Path(str(p)))

    def _human_bytes(n: int) -> str:
        units = ["B", "KB", "MB", "GB", "TB"]
        x = float(n)
        for u in units:
            if x < 1024.0 or u == units[-1]:
                return f"{x:.2f}{u}"
            x /= 1024.0
        return f"{x:.2f}TB"

    def _file_info(path: Path | None, X: np.ndarray, y: np.ndarray) -> str:
        if path is None or not path.exists():
            return "file=NA"

        size_disk = path.stat().st_size
        raw_est = int(getattr(X, "nbytes", 0) + getattr(y, "nbytes", 0))
        ratio = (size_disk / raw_est) if raw_est > 0 else float("nan")

        if raw_est > 0 and ratio < 0.85:
            kind = "npz-compressed (probable)"
        else:
            kind = "npz (sin compresión o compresión mínima)"

        return (
            f"file={_human_bytes(size_disk)} | "
            f"raw~{_human_bytes(raw_est)} | "
            f"{kind} | ratio={ratio:.3f}"
        )

    def _nan_info(arr) -> str:
        try:
            n_nan = int(pd.isna(arr).sum())
            return f"NaN={n_nan}"
        except Exception:
            return "NaN=NA"

    def _unique_y_info(y) -> str:
        try:
            vals = np.unique(y)
            if len(vals) <= 10:
                return f"classes={vals.tolist()}"
            return f"classes={len(vals)} valores únicos"
        except Exception:
            return "classes=NA"

    # ------------------------------------------------------------
    # 2) Paths opcionales
    # ------------------------------------------------------------
    p_train = _to_path(path_train)
    p_valid = _to_path(path_valid)
    p_test  = _to_path(path_test)

    # ------------------------------------------------------------
    # 3) Impresión por split
    # ------------------------------------------------------------
    def _print_line(name, X, y, path: Path | None):
        if not hasattr(X, "shape") or not hasattr(y, "shape"):
            print(f"{name} | ERROR: X o y sin shape")
            return

        N = X.shape[0] if X.ndim >= 1 else 0

        # --------------------------------------------------------
        # Validación de dimensiones esperadas
        # --------------------------------------------------------
        if flatten:
            if X.ndim == 2:
                dim = X.shape[1]
                expected = window_size * n_features
                x_ok = (dim == expected)
                dim_info = f"dim={dim}"
            else:
                x_ok = False
                dim_info = "dim=?"
        else:
            if X.ndim == 3:
                L, F = X.shape[1], X.shape[2]
                x_ok = (L == window_size) and (F == n_features)
                dim_info = f"(L,F)=({L},{F})"
            else:
                x_ok = False
                dim_info = "dim=?"

        # --------------------------------------------------------
        # Validación de y
        # --------------------------------------------------------
        y_ok = (
            (y.ndim == 1 and y.shape[0] == N) or
            (y.ndim == 2 and y.shape == (N, 1))
        )

        status = "OK" if x_ok and y_ok else "ERROR"

        # --------------------------------------------------------
        # dtype / NaN / memoria
        # --------------------------------------------------------
        x_dtype = getattr(X, "dtype", "NA")
        y_dtype = getattr(y, "dtype", "NA")
        x_mem = _human_bytes(getattr(X, "nbytes", 0))
        y_mem = _human_bytes(getattr(y, "nbytes", 0))
        x_nan = _nan_info(X)
        y_nan = _nan_info(y)
        y_classes = _unique_y_info(y)

        # --------------------------------------------------------
        # Tamaño archivo
        # --------------------------------------------------------
        finfo = _file_info(path, X, y)

        print(
            f"{name:<35} | "
            f"X:{X.shape} ({x_dtype}) | "
            f"y:{y.shape} ({y_dtype}) | "
            f"N={N} | "
            f"{dim_info} | "
            f"{status} | "
            f"RAM X={x_mem} y={y_mem} | "
            f"{x_nan} {y_nan} | "
            f"{y_classes} | "
            f"{finfo}"
        )

    # ------------------------------------------------------------
    # 4) Salida final por split
    # ------------------------------------------------------------
    _print_line(f"[{target_col} | L={window_size} | train]", X_train, y_train, p_train)
    _print_line(f"[{target_col} | L={window_size} | valid]", X_valid, y_valid, p_valid)
    _print_line(f"[{target_col} | L={window_size} | test ]", X_test,  y_test,  p_test)
    print()

### **4.2.4. Utilidades**

In [69]:
features_to_windows = [
    'regime_id',
    'roc_30', 'roc_60',
    'stoch_k_30', 'atr_norm_10',
    ]

n_features = len(features_to_windows)
n_features

print (f'features_to_windows ({n_features}): {features_to_windows}')
print(f'targets_col: {targets_col}')
print(f'window_sizes: {window_sizes}')

features_to_windows (5): ['regime_id', 'roc_30', 'roc_60', 'stoch_k_30', 'atr_norm_10']
targets_col: ['t2_dir_thr_90', 't2_dir_thr_120']
window_sizes: [30, 60, 90, 120, 180]


In [74]:
# Mapa target -> datasets escalados
targets_windows = {
    targets_col[0]: (mnq_t2_train_s, mnq_t2_valid_s, mnq_t2_test_s),
    targets_col[1]: (mnq_t2_train_s, mnq_t2_valid_s, mnq_t2_test_s),
}

In [77]:
OUT_WINDOWS_SEQ2ONE_DIR = DRIVE_DIR / "data/07_windows/seq2one"

### **4.2.5. Cargar ventanas .npZ**

In [82]:
def _load_npz(path_npz: Path):
    """
    Carga arrays X e y desde un archivo .npz.

    Validaciones:
    - Verifica que el archivo exista.
    - Verifica que el .npz contenga claves 'X' e 'y'.
    - Mantiene dtype consistente con lo guardado.
    """

    if not path_npz.exists():
        raise FileNotFoundError(f"No se encontró el archivo NPZ: {path_npz}")

    with np.load(path_npz, allow_pickle=False) as z:

        if "X" not in z:
            raise KeyError(f"NPZ inválido (falta clave 'X'): {path_npz}")

        if "y" not in z:
            raise KeyError(f"NPZ inválido (falta clave 'y'): {path_npz}")

        X = z["X"]
        y = z["y"]

    # Normalización mínima del pipeline
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y)

    return X, y

### **4.2.6. Generación de resumen de ventanas `seq2one`**

In [86]:
import json
from pathlib import Path
import numpy as np
from datetime import datetime

def _human_bytes(n: int | None) -> str | None:
    if n is None:
        return None

    units = ["B", "KB", "MB", "GB", "TB"]
    x = float(n)

    for u in units:
        if x < 1024.0 or u == units[-1]:
            return f"{x:.2f}{u}"
        x /= 1024.0

    return f"{x:.2f}TB"


def _split_file_info(path: Path, X: np.ndarray, y: np.ndarray) -> dict:
    """
    Devuelve metadata estructural de un split de ventanas.

    Incluye:
    - tamaño real del .npz en disco
    - tamaño bruto estimado en RAM (X.nbytes + y.nbytes)
    - ratio de compresión aproximado
    - shapes y dtypes
    """
    path = Path(path)
    exists = path.exists()

    size_disk = path.stat().st_size if exists else None
    raw_est = int(getattr(X, "nbytes", 0) + getattr(y, "nbytes", 0))
    ratio = (size_disk / raw_est) if (size_disk is not None and raw_est > 0) else None

    if ratio is None:
        kind = None
    else:
        kind = (
            "npz-compressed (probable)"
            if ratio < 0.85
            else "npz (sin compresión o mínima)"
        )

    return {
        "file_exists": exists,
        "file_name": path.name,
        "file_path": str(path),
        "size_disk_bytes": size_disk,
        "size_disk_human": _human_bytes(size_disk),
        "raw_estimated_bytes": raw_est,
        "raw_estimated_human": _human_bytes(raw_est),
        "compression_ratio": round(ratio, 6) if ratio is not None else None,
        "compression_kind": kind,
        "shape_X": list(X.shape),
        "shape_y": list(y.shape),
        "dtype_X": str(getattr(X, "dtype", "NA")),
        "dtype_y": str(getattr(y, "dtype", "NA")),
    }

In [87]:
def update_seq2one_manifest(
    manifest: dict,
    *,
    target_col: str,
    window_size: int,
    n_features: int,
    features: list[str],
    X_train, y_train,
    X_valid, y_valid,
    X_test, y_test,
    path_train: Path,
    path_valid: Path,
    path_test: Path,
    date_col: str = "date",
    minute_col: str = "minute_of_day",
    flatten: bool = False,
    drop_windows_with_nan: bool = True,
) -> dict:
    """
    Actualiza el manifest global con la entrada (target_col, window_size).

    Retorna el manifest actualizado.
    La mutación también ocurre in-place sobre manifest.
    """

    # ------------------------------------------------------------
    # 1. Asegurar estructura base del manifest
    # ------------------------------------------------------------
    if "items" not in manifest:
        manifest["items"] = {}

    # ------------------------------------------------------------
    # 2. Clave única por configuración
    # ------------------------------------------------------------
    key = f"{target_col}__L{window_size}"

    # ------------------------------------------------------------
    # 3. Normalizar arrays
    # ------------------------------------------------------------
    X_train_np = np.asarray(X_train)
    y_train_np = np.asarray(y_train)

    X_valid_np = np.asarray(X_valid)
    y_valid_np = np.asarray(y_valid)

    X_test_np = np.asarray(X_test)
    y_test_np = np.asarray(y_test)

    # ------------------------------------------------------------
    # 4. Actualizar manifest
    # ------------------------------------------------------------
    manifest["items"][key] = {
        "target": target_col,
        "window_size": int(window_size),
        "n_features": int(n_features),
        "features": list(features),
        "date_col": date_col,
        "minute_col": minute_col,
        "flatten": bool(flatten),
        "drop_windows_with_nan": bool(drop_windows_with_nan),
        "format_expected": "npz_compressed",
        "updated_at": datetime.utcnow().isoformat() + "Z",
        "splits": {
            "train": _split_file_info(Path(path_train), X_train_np, y_train_np),
            "valid": _split_file_info(Path(path_valid), X_valid_np, y_valid_np),
            "test":  _split_file_info(Path(path_test),  X_test_np,  y_test_np),
        },
    }

    return manifest

In [81]:
from pathlib import Path
import json
from datetime import datetime


def save_seq2one_manifest(manifest: dict, *, out_path: Path) -> Path:
    """
    Guarda el manifest global de ventanas seq2one en formato JSON.

    - Crea directorios si no existen.
    - Agrega timestamp de guardado.
    - Ordena claves para reproducibilidad.
    """

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    # Asegurar estructura mínima
    if "items" not in manifest:
        manifest["items"] = {}

    # Timestamp global
    manifest["saved_at"] = datetime.utcnow().isoformat() + "Z"

    # Guardado
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(
            manifest,
            f,
            indent=2,
            ensure_ascii=False,
            sort_keys=True
        )

    print(f"[MANIFEST] Guardado: {out_path}")

    return out_path

### **4.2.7. Generación de ventanas `seq2one`**

In [88]:
from pathlib import Path
from datetime import datetime

# ============================================================
# Manifest global
# ============================================================
manifest = {
    "schema": "seq2one_windows_manifest_v1",
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "items": {}
}

# ============================================================
# Loop principal
# ============================================================
for target_col, (df_tr, df_va, df_te) in targets_windows.items():
    for L in window_sizes:

        print("\n" + "=" * 70)
        print(f"SEQ2ONE | TARGET={target_col} | L={L} | F={len(features_to_windows)}")
        print("=" * 70)

        # --------------------------------------------------------
        # Directorio por tamaño de ventana
        # --------------------------------------------------------
        out_dir_L = OUT_WINDOWS_SEQ2ONE_DIR / f"L{L}"
        out_dir_L.mkdir(parents=True, exist_ok=True)

        # --------------------------------------------------------
        # Archivos NPZ por split
        # --------------------------------------------------------
        out_train = out_dir_L / f"windows_{target_col}_train.npz"
        out_valid = out_dir_L / f"windows_{target_col}_valid.npz"
        out_test  = out_dir_L / f"windows_{target_col}_test.npz"

        # --------------------------------------------------------
        # Metadata opcional por split
        # --------------------------------------------------------
        out_meta_train = out_dir_L / f"meta_{target_col}_train.json"
        out_meta_valid = out_dir_L / f"meta_{target_col}_valid.json"
        out_meta_test  = out_dir_L / f"meta_{target_col}_test.json"

        # --------------------------------------------------------
        # Cargar o construir ventanas
        # --------------------------------------------------------
        X_train, y_train, X_valid, y_valid, X_test, y_test = prepare_or_load_seq2one_windows_npz(
            mnq_train=df_tr,
            mnq_valid=df_va,
            mnq_test=df_te,
            features=features_to_windows,
            target_col=target_col,
            window_size=L,
            out_train=out_train,
            out_valid=out_valid,
            out_test=out_test,
            out_meta_train=out_meta_train,
            out_meta_valid=out_meta_valid,
            out_meta_test=out_meta_test,
            date_col="date",
            minute_col="minute_of_day",
            flatten=False,
            drop_windows_with_nan=True,
            verbose=True,
            repair_axis_order_if_needed=True,
            validate_temporal_order=True,
            validate_unique_minutes=True,
        )

        # --------------------------------------------------------
        # Auditoría compacta de ventanas
        # --------------------------------------------------------
        xy_info_seq2one_compact(
            target_col=target_col,
            X_train=X_train,
            y_train=y_train,
            X_valid=X_valid,
            y_valid=y_valid,
            X_test=X_test,
            y_test=y_test,
            window_size=L,
            n_features=len(features_to_windows),
            flatten=False,
            path_train=out_train,
            path_valid=out_valid,
            path_test=out_test,
        )

        # --------------------------------------------------------
        # Actualizar manifest global
        # --------------------------------------------------------
        manifest = update_seq2one_manifest(
            manifest=manifest,
            target_col=target_col,
            window_size=L,
            n_features=len(features_to_windows),
            features=features_to_windows,
            X_train=X_train,
            y_train=y_train,
            X_valid=X_valid,
            y_valid=y_valid,
            X_test=X_test,
            y_test=y_test,
            path_train=out_train,
            path_valid=out_valid,
            path_test=out_test,
            date_col="date",
            minute_col="minute_of_day",
            flatten=False,
            drop_windows_with_nan=True,
        )

# ============================================================
# Guardar manifest global
# ============================================================
save_seq2one_manifest(
    manifest,
    out_path=OUT_WINDOWS_SEQ2ONE_DIR / "seq2one_windows_manifest.json",
)


SEQ2ONE | TARGET=t2_dir_thr_90 | L=30 | F=5
[t2_dir_thr_90 | L=30 | train] BUILD  X=(436692, 30, 5)  y=(436692,)  -> windows_t2_dir_thr_90_train.npz
[t2_dir_thr_90 | L=30 | valid] BUILD  X=(93508, 30, 5)  y=(93508,)  -> windows_t2_dir_thr_90_valid.npz
[t2_dir_thr_90 | L=30 | test] BUILD  X=(93990, 30, 5)  y=(93990,)  -> windows_t2_dir_thr_90_test.npz
[t2_dir_thr_90 | L=30 | train]      | X:(436692, 30, 5) (float32) | y:(436692,) (int8) | N=436692 | (L,F)=(30,5) | OK | RAM X=249.88MB y=426.46KB | NaN=0 NaN=0 | classes=[-1, 0, 1] | file=8.96MB | raw~250.29MB | npz-compressed (probable) | ratio=0.036
[t2_dir_thr_90 | L=30 | valid]      | X:(93508, 30, 5) (float32) | y:(93508,) (int8) | N=93508 | (L,F)=(30,5) | OK | RAM X=53.51MB y=91.32KB | NaN=0 NaN=0 | classes=[-1, 0, 1] | file=1.92MB | raw~53.59MB | npz-compressed (probable) | ratio=0.036
[t2_dir_thr_90 | L=30 | test ]      | X:(93990, 30, 5) (float32) | y:(93990,) (int8) | N=93990 | (L,F)=(30,5) | OK | RAM X=53.78MB y=91.79KB | NaN=0

PosixPath('/content/drive/MyDrive/neural_profit/data/07_windows/seq2one/seq2one_windows_manifest.json')

In [89]:
manifest

{'schema': 'seq2one_windows_manifest_v1',
 'created_at': '2026-03-24T03:05:10',
 'items': {'t2_dir_thr_90__L30': {'target': 't2_dir_thr_90',
   'window_size': 30,
   'n_features': 5,
   'features': ['regime_id', 'roc_30', 'roc_60', 'stoch_k_30', 'atr_norm_10'],
   'date_col': 'date',
   'minute_col': 'minute_of_day',
   'flatten': False,
   'drop_windows_with_nan': True,
   'format_expected': 'npz_compressed',
   'updated_at': '2026-03-24T03:05:15.127456Z',
   'splits': {'train': {'file_exists': True,
     'file_name': 'windows_t2_dir_thr_90_train.npz',
     'file_path': '/content/drive/MyDrive/neural_profit/data/07_windows/seq2one/L30/windows_t2_dir_thr_90_train.npz',
     'size_disk_bytes': 9396782,
     'size_disk_human': '8.96MB',
     'raw_estimated_bytes': 262451892,
     'raw_estimated_human': '250.29MB',
     'compression_ratio': 0.035804,
     'compression_kind': 'npz-compressed (probable)',
     'shape_X': [436692, 30, 5],
     'shape_y': [436692],
     'dtype_X': 'float32',


Las conclusiones principales son estas:

1. Se registraron correctamente las **10 configuraciones esperadas**: 2 targets (`t2_dir_thr_90`, `t2_dir_thr_120`) × 5 tamaños de ventana (`30, 60, 90, 120, 180`).

2. Las shapes de `X` e `y` son consistentes con el esquema **seq2one**:

   * `X`: `(N, L, 5)`
   * `y`: `(N,)`
     Esto confirma que la generación de ventanas quedó estructuralmente correcta.

3. El número de muestras por split y por ventana coincide con lo que ya habías validado antes en el análisis de viabilidad. No se observan inconsistencias entre el manifest y los conteos esperados.

4. Los archivos `.npz` existen en todas las configuraciones y el manifest guarda bien la trazabilidad:

   * target
   * window size
   * features usadas
   * rutas
   * shapes
   * dtypes
   * tamaño en disco
   * ratio de compresión.

5. La compresión es muy buena en todos los casos. Los ratios son bajos, por lo que el almacenamiento quedó eficiente.

6. Dado que el target corresponde a clases discretas, es correcto y consistente representarlo como un tipo entero. En este caso, el uso de `int8` para `y` es apropiado, ya que preserva la naturaleza categórica del target y optimiza el uso de memoria sin pérdida de información.


7. Que `regime_id` aparezca dentro de las `features` del manifest es correcto. No se escaló, pero sí forma parte de la entrada del modelo, así que debe figurar en las ventanas.



# **5. Alineamiento con libro de ML**

**Preprocesamiento y escalamiento de datos**

Este paso se realiza **después del split temporal y antes de la generación de ventanas**, siguiendo las buenas prácticas para modelado de series temporales financieras.

---

**Aspectos correctamente alineados**

1. Orden del proceso

* El split temporal se realiza antes de cualquier transformación.
* El escalamiento se aplica sobre los datasets tabulares.
* La generación de ventanas se realiza posteriormente sobre los datos ya escalados.

2. Regla crítica anti-leakage

* El scaler se ajusta exclusivamente con el conjunto de entrenamiento (TRAIN).
* Los conjuntos de validación y test se transforman utilizando ese mismo scaler, sin volver a ajustarlo.

3. Escalamiento aplicado únicamente a features

* Solo se escalan las variables de entrada seleccionadas (`roc_30`, `roc_60`, `stoch_k_30`, `atr_norm_10`).
* Variables categóricas como `regime_id` no se escalan.
* El target (`t2_dir_thr_90`, `t2_dir_thr_120`) no se transforma, preservando su naturaleza discreta.

4. Consistencia entre targets

* El mismo scaler se aplica independientemente del target utilizado.
* No se mezclan transformaciones entre configuraciones, garantizando comparabilidad entre experimentos.

5. Persistencia

* El scaler es guardado y reutilizado en todo el pipeline.
* Se garantiza reproducibilidad entre entrenamiento, validación, test e inferencia.

---

**Criterio de escalamiento**

Se utiliza StandardScaler (z-score), ajustado exclusivamente con el conjunto de entrenamiento.

El escalamiento se realiza por feature de forma global (no por día), permitiendo capturar la distribución completa del dataset sin introducir leakage temporal.

---

**Verificación automática del escalamiento**

Se realizó una validación sobre los datasets escalados, verificando que:

* la media por feature en TRAIN sea aproximadamente 0,
* la desviación estándar por feature en TRAIN sea aproximadamente 1,
* VALID y TEST hayan sido correctamente transformados,
* el target no haya sido modificado,
* la estructura temporal (orden, índice y alineación) se mantenga intacta.

Los resultados confirman que el escalamiento fue aplicado correctamente, sin introducir data leakage y preservando la integridad temporal del dataset.

---

**Conclusión**

El proceso de preprocesamiento y escalamiento se encuentra correctamente implementado y alineado con las buenas prácticas del libro.

Se garantiza que los datos utilizados para el modelado:

* respetan la causalidad temporal,
* mantienen consistencia entre splits,
* y se encuentran en una escala adecuada para el entrenamiento de modelos.


# **Conclusión global del stage**



1. Definición de targets

   Se definieron targets de tipo T2 (`t2_dir_thr_90` y `t2_dir_thr_120`) que incorporan un criterio de umbral y reducen el ruido inherente de los retornos financieros. Estos targets presentan estructura, magnitud económica y mejor interpretabilidad que los enfoques basados en regresión directa, alineándose con las recomendaciones del libro.

2. Selección de features

   Se construyó un conjunto reducido y robusto de variables (`roc_30`, `roc_60`, `stoch_k_30`, `atr_norm_10`, `regime_id`) basado en señal predictiva, consistencia out-of-sample y baja redundancia. Esto permite un balance adecuado entre capacidad explicativa y control del overfitting.

3. Split temporal

   La partición en train, validation y test se realizó respetando el orden cronológico por días, evitando completamente el data leakage y garantizando una evaluación realista del modelo en datos futuros.

4. Escalamiento

   El escalado fue aplicado correctamente sobre las features continuas, ajustando el scaler únicamente con el conjunto de entrenamiento y manteniendo intactos tanto el target como las variables categóricas. Se verificó que no se introdujo contaminación entre splits y que la estructura temporal se preserva completamente.

5. Generación de ventanas

   Se implementó un esquema seq2one consistente, donde cada ventana utiliza únicamente información pasada y el target corresponde a la última observación de la secuencia.
   La generación se realizó por día, evitando mezcla entre sesiones y respetando la causalidad temporal.
   Se construyeron correctamente todas las combinaciones de ventanas para ambos targets y múltiples tamaños (`L = 30, 60, 90, 120, 180`).

6. Integridad del pipeline

   Se validó que:

    * no existen NaNs en las ventanas finales,
    * las dimensiones de entrada y salida son correctas,
    * los dtypes son consistentes (`X: float32`, `y: int8`),
    * los datasets se encuentran ordenados temporalmente,
    * y todos los artefactos (datasets, scaler, ventanas, manifest) son reproducibles.

7. Trazabilidad y reproducibilidad

   Se implementó un sistema de persistencia completo que incluye:

    * datasets escalados,
    * ventanas en formato `.npz`,
    * metadata por split,
    * y un manifest global que documenta todas las configuraciones.

    Esto permite reconstruir cualquier experimento de forma determinista.

8. Alineación con el enfoque del libro

   El pipeline sigue las buenas prácticas de *Machine Learning for Trading*:

    * separación estricta entre train y evaluación,
    * control explícito del leakage,
    * uso de targets más robustos que retornos crudos,
    * y preparación adecuada de datos para modelado secuencial.

---

**Conclusión final**

El dataset final y su representación en ventanas están correctamente construidos, libres de leakage y listos para ser utilizados en el entrenamiento de modelos.

El pipeline garantiza consistencia, robustez y reproducibilidad, por lo que el siguiente paso —modelado— puede abordarse con una base sólida y bien validada.
